In [ ]:
import datetime as dt

import geopandas as gpd
import pandas as pd

In [ ]:
spatial_extent_filepath = "../data/processed/bern-extent.gpkg"

ts_df_filepath = "../data/raw/ucb/ts-df-jja-2023-2024.csv"
stations_gdf_filepath = "../data/raw/ucb/metadata_network_2023_2024.csv"

# ts_df_time_col = "Time_UTC"
stations_id_col = "Log_NR"
stations_src_crs = "epsg:4326"
stations_x_col = "OST_CHTOPO"
stations_y_col = "NORD_CHTOPO"

# study period (e.g., JJA)
start_year = 2023
end_year = 2024
start_month = 6
end_month = 8

# output params
dst_crs = "epsg:2056"

# output files
dst_ts_df_filepath = "../data/interim/bern-lcd-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/bern-lcd-stations.gpkg"

In [ ]:
ts_df = pd.read_csv(ts_df_filepath, parse_dates=["time"], index_col="time")
# filter by study period
# note that we need to resample each year data before concat
# otherwise resample would add nan rows for all the missing non-JJA data
ts_df = pd.concat(
    [
        ts_df[
            pd.Series(ts_df.index)
            .between(
                pd.Timestamp(year=year, month=start_month, day=1),
                # get latest moment of the latest day of the month
                # ACHTUNG: this won't work if `end_month` is 12 (see commented code
                # below)
                pd.Timestamp(
                    dt.datetime.combine(
                        dt.date(year, end_month + 1, 1) - dt.timedelta(days=1),
                        dt.time.max,
                    ),
                ),
            )
            .values
        ]
        .resample("h")
        .mean()
        for year in range(start_year, end_year + 1)
    ]
)
# show the data frame
ts_df.head()

,1,10,101,102,111,112,113,116,117,124,...,83,86,87,89,9,98,99,134,135,52
time,,,,,,,,,,,,,,,,,,,,,
2023-06-01 00:00:00,13.245212,13.936383,NaN,13.608377,13.985339,12.842438,NaN,12.282114,10.598026,13.898553,...,13.591020,13.226965,12.413405,14.421047,11.615867,11.760510,12.112103,NaN,NaN,NaN
2023-06-01 01:00:00,12.611009,13.336894,NaN,13.129943,13.643091,12.456130,NaN,11.706658,10.223735,13.460174,...,13.403652,12.958597,12.059141,14.175822,11.211312,11.408026,11.638120,NaN,NaN,NaN
2023-06-01 02:00:00,12.210460,12.980850,NaN,12.857570,13.356921,12.036889,NaN,11.239796,10.299840,13.309300,...,13.054284,12.549147,11.878004,13.448157,10.942054,11.149449,11.464548,NaN,NaN,NaN
2023-06-01 03:00:00,12.405839,12.821520,NaN,12.674652,13.068526,11.904707,NaN,10.979438,10.421340,12.924328,...,12.767669,12.190878,11.756504,13.238092,10.754686,10.793406,11.027949,NaN,NaN,NaN
2023-06-01 04:00:00,12.718713,12.669757,NaN,12.507757,12.944355,12.266982,12.611454,10.997241,10.390631,12.494405,...,12.864246,12.071603,11.703543,13.471300,11.058658,11.222438,11.245581,NaN,NaN,NaN


In [ ]:
# region_gser = gpd.read_file(spatial_extent_filepath)["geometry"]
stations_gdf = pd.read_csv(stations_gdf_filepath, sep=";", encoding="latin1")
stations_gdf = gpd.GeoDataFrame(
    stations_gdf,
    geometry=gpd.points_from_xy(
        stations_gdf[stations_x_col], stations_gdf[stations_y_col]
    ),
    crs=stations_src_crs,
).to_crs(dst_crs)  # .to_crs(region_gser.crs)
# # filter by spatial extent
# stations_gdf = stations_gdf[stations_gdf.within(region_gser.iloc[0])]
# rename id column and set it as index
stations_gdf = stations_gdf.rename(columns={stations_id_col: "station_id"}).set_index(
    "station_id"
)
stations_gdf.index = stations_gdf.index.astype(str)
# show the geo-data frame
stations_gdf.head()

,NAME_old,NAME_new,NORD_CHTOPO,OST_CHTOPO,LV_03_E,LV_03_N,HüM,geometry
station_id,,,,,,,,
1,VonRoll PH,VonRoll,46.95291,7.42252,2598773.70,1200203.5,553.9,POINT (2598773.471 1200203.276)
2,Europaplatz,Europaplatz,46.94333,7.40633,2597540.60,1199138.8,555.5,POINT (2597540.582 1199138.657)
7,Kreuzung Brunmatt- Schwarztorstrasse,Brunnhof,46.94479,7.42713,2599124.40,1199300.1,528.6,POINT (2599124.256 1199300.504)
8,Wankdorf ESP,Wankdorf ESP,46.96800,7.46247,2601814.20,1201881.4,550.5,POINT (2601814.111 1201880.959)
9,Bremgartenfriedhof,Bremgartenfriedhof,46.95030,7.42015,2598592.61,1199912.8,552.4,POINT (2598592.987 1199913.163)


In [ ]:
# save only stations for which we have both data and metadata
valid_stations = ts_df.columns.intersection(stations_gdf.index)
# save the time series of measurements to a file
ts_df[valid_stations].to_csv(dst_ts_df_filepath)
# save the stations' locations to a file
stations_gdf.loc[valid_stations].rename_axis(index="station_id").to_file(
    dst_stations_gdf_filepath
)